In [ ]:
_repo_root = !git rev-parse --show-toplevel
%cd {_repo_root[0]}
del _repo_root

# Criteo scaling

Validation-selected test performance under repeated-shuffle Adam/SparseAdam training. The plots separate per-feature parameter-width comparison, preprocessing comparison, and dimension scaling. Lower is better.

## Setup

In [ ]:
import warnings

import matplotlib as mpl
import pandas as pd
from tqdm import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)

from paper.experiments.criteo_scaling import (
    PROFILES,
    default_raw_path,
    select_lr,
    summarize_raw,
    validate_raw,
)
from paper.plotting import (
    plot_criteo_models_by_dimension,
    plot_criteo_fm_dimensions,
    plot_criteo_spectral_comparison,
    plot_criteo_spectral_dimensions,
)

mpl.rcParams["figure.dpi"] = 140

In [ ]:
profile_name = "full"
profile = PROFILES[profile_name]
raw_path = default_raw_path(profile_name)
metric = "logloss"

## Load and validate results

Preprocessing is fitted once on a fixed 10% sample of the training pool. Each data seed defines a deterministic stream of fresh permutations over that fixed pool, and `train_size` counts cumulative optimizer examples rather than distinct rows. The explicit profile `train_sizes` are the only evaluation checkpoints and their maximum is the final budget; the stream creates as many permutations as that budget requires. Learning rates are selected independently at each checkpoint using validation loss; test metrics are then reported only for the selected checkpoint trajectories.

In [ ]:
if not raw_path.exists():
    raise FileNotFoundError(
        f"{raw_path} does not exist; run paper.experiments.criteo_scaling first"
    )

raw = pd.read_csv(raw_path)
validate_raw(raw, profile)

selected = select_lr(raw)
summary = summarize_raw(raw)
summary[[
    "train_size",
    "model",
    "dim",
    "selected_lr",
    "median_test_logloss",
    "median_test_brier",
    "n",
]].sort_values(["train_size", "dim", "model"])

## All models within each dimension

Each facet compares linear, linear-new, FM, spectral-old, and spectral-new. FM rank is matched to the spectral matrix parameter width per encoded feature; bucket and hybrid preprocessing have different vocabulary sizes, so their total parameter counts are not equal. The two dimensionless linear curves are repeated for reference.

In [ ]:
plot_criteo_models_by_dimension(selected, metric=metric);

## Spectral preprocessing within each dimension

This view removes the linear and FM references to isolate the effect of the numerical representation.

In [ ]:
plot_criteo_spectral_comparison(selected, metric=metric);

## Spectral-new across dimensions

All hybrid-preprocessed spectral capacities on one axis.

In [ ]:
plot_criteo_spectral_dimensions(
    selected, "spectral-new", metric=metric
);

### Spectral-new detail from $2^{18}$ to $2^{24}$

In [ ]:
plot_criteo_spectral_dimensions(
    selected,
    "spectral-new",
    metric=metric,
    xlim=(2**18, 2**24),
);

## FM across embedding dimensions

Compare the per-feature-width-matched FM embedding ranks across the full scaling range.

In [ ]:
plot_criteo_fm_dimensions(selected, metric=metric);

## Spectral-old across dimensions

All bucket-preprocessed spectral capacities on one axis.

In [ ]:
plot_criteo_spectral_dimensions(
    selected, "spectral-old", metric=metric
);

## Reading the comparison

The first two figures hold dimension fixed, so old and new spectral neurons have the same matrix width per encoded feature; their different preprocessing vocabularies can still produce different total parameter counts. The spectral dimension views hold preprocessing fixed, while the FM view varies embedding rank. Lines are medians across retraining runs and bands span the 25th to 75th percentiles.

The full profile evaluates only its requested powers of two through $2^{26}$ and stops there. The training-pool size determines how many fresh deterministic permutations are needed for each data seed, but pass boundaries do not add evaluation checkpoints or change the budget. Batches are formed over the concatenated stream, so a batch may cross a pass boundary.

These are fixed-representation examples-seen curves: preprocessing and model size stay fixed while optimizer exposure grows, and rows repeat after the first pass. They are not end-to-end dataset-size scaling curves. Change `metric` in Setup to `"brier"` for the corresponding Brier-score views.